# 02. Construct PheCode + measurement dataset

Adapted from Naomi's `construct_phecode_measurement_dataset.ipynb`.

This notebook uses the precomputed `patient_index.parquet` from notebook 01
(psychosis-risk cases + non-psychosis controls, each with a fixed `index_datetime`).

The downstream intermediate file names are intentionally unchanged.

Outputs (in `output/`):
- `04_demographics_features.parquet`
- `05_selected_measurements.parquet`
- `06_measurement_features_long.parquet`
- `07_feature_matrix.parquet`     <- X for notebook 03
- `08_disease_matrix.parquet`     <- Y for notebook 03
- summary CSVs

**Design note on Y:** This is a feature-extraction / cohort-characterization task,
not a pure prediction task. Psychosis-related PheCodes are therefore retained in Y
unless they are removed by the curated review exclusion file. Y represents each
patient's phenotype profile, while X is built from demographics and measurements
up to `index_datetime`.


## 1. Setup

In [1]:
# !pip install duckdb pyarrow pandas numpy
from pathlib import Path
import json
import re

import duckdb
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

con = duckdb.connect()
print('DuckDB version:', duckdb.__version__)


DuckDB version: 1.5.1


## 2. Configuration

In [2]:
DATA_PATH = '/home/jupyter/2836994-data2'
PHECODE_MAP_PATH = 'phecode_icd10.csv'
EXCLUDED_LABELS_PATH = 'final_raw_labels_to_exclude_union_phecode_name_review.txt'
PATIENT_INDEX_PATH = 'output/patient_index.parquet'

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# LOOKBACK_DAYS = 1000
MIN_LABEL_PATIENTS = 0
MAX_LABELS = 2000
MIN_MEASUREMENT_PERSON_PERCENT = 0.005
MIN_MEASUREMENT_PATIENTS = None
MAX_MEASUREMENTS = 10000
# REQUIRE_MEASUREMENT_IN_LOOKBACK = True

# Measurement concepts to EXCLUDE as features (real target leakage from Naomi's notebook).
# 42538860  Charlson Comorbidity Index -- weighted sum of diagnoses (direct leakage to PheCode Y)
# 37016720  Hospital readmission risk score -- composite of diagnoses
# These are scores COMPUTED FROM diagnoses, not real measurements, so they leak
# regardless of task (prediction or feature extraction). Keep these excluded.
EXCLUDED_MEASUREMENT_CONCEPT_IDS = [42538860, 37016720]
EXCLUDED_CONCEPT_SQL = (
    ' AND m.measurement_concept_id NOT IN ('
    + ','.join(str(x) for x in EXCLUDED_MEASUREMENT_CONCEPT_IDS) + ')'
    if EXCLUDED_MEASUREMENT_CONCEPT_IDS else ''
)


## 3. Helper functions

In [3]:
def parquet_glob(table):
    return str(Path(DATA_PATH) / table / '*.parquet')

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def qident(value):
    return '"' + str(value).replace('"', '""') + '"'

def save_table(table_name, file_name):
    out = OUTPUT_DIR / file_name
    con.execute(f'COPY {table_name} TO {sql_literal(str(out))} (FORMAT PARQUET)')
    print('wrote', out)
    return out

def save_query_csv(query, file_name):
    out = OUTPUT_DIR / file_name
    con.sql(query).to_df().to_csv(out, index=False)
    print('wrote', out)
    return out

person_path = parquet_glob('person')
measurement_path = parquet_glob('measurement')
condition_path = parquet_glob('condition_occurrence')
concept_path = parquet_glob('concept')

print('person:', person_path)
print('measurement:', measurement_path)
print('condition_occurrence:', condition_path)


person: /home/jupyter/2836994-data2/person/*.parquet
measurement: /home/jupyter/2836994-data2/measurement/*.parquet
condition_occurrence: /home/jupyter/2836994-data2/condition_occurrence/*.parquet


## 4. Load and normalize ICD-10 to PheCode mapping

In [4]:
phe = pd.read_csv(PHECODE_MAP_PATH, dtype=str)
phe.columns = [c.strip() for c in phe.columns]

required_cols = {'ALT_CODE', 'ICD10', 'PheCode', 'Phenotype'}
missing = required_cols - set(phe.columns)
if missing:
    raise ValueError(f'Missing expected columns in {PHECODE_MAP_PATH}: {missing}')

def normalize_icd10(x):
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    x = re.sub(r'[^A-Z0-9]', '', x)
    return x or None

def clean_phecode(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if not x or x.lower() == 'nan':
        return None
    return x

def format_phecode_label(phecode, phenotype):
    phecode = clean_phecode(phecode)
    if phecode is None:
        return None
    phenotype = '' if pd.isna(phenotype) else str(phenotype).strip()
    phenotype = re.sub(r'[^A-Za-z0-9]', '_', phenotype)
    return f'{phecode}__{phenotype}'

map_rows = []
for _, row in phe.iterrows():
    phecode = clean_phecode(row['PheCode'])
    if phecode is None:
        continue
    for source_col in ['ALT_CODE', 'ICD10']:
        code_norm = normalize_icd10(row[source_col])
        if code_norm:
            map_rows.append({
                'icd10_norm': code_norm,
                'phecode': phecode,
                'phenotype': row['Phenotype'],
                'disease_label': format_phecode_label(phecode, row['Phenotype']),
            })

phe_map = (pd.DataFrame(map_rows)
           .drop_duplicates()
           .dropna(subset=['icd10_norm', 'phecode', 'disease_label']))
con.register('phe_map_df', phe_map)
con.execute('CREATE OR REPLACE TEMP TABLE phecode_map AS SELECT DISTINCT * FROM phe_map_df')

display(phe_map.head())
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT icd10_norm) AS n_icd10_norm,
       COUNT(DISTINCT phecode) AS n_phecodes
FROM phecode_map
''').to_df())


,icd10_norm,phecode,phenotype,disease_label
0,A00,8.0,Intestinal infection,8.0__Intestinal_infection
2,A000,8.0,Intestinal infection,8.0__Intestinal_infection
4,A001,8.0,Intestinal infection,8.0__Intestinal_infection
6,A009,8.0,Intestinal infection,8.0__Intestinal_infection
8,A01,8.0,Intestinal infection,8.0__Intestinal_infection


,n_rows,n_icd10_norm,n_phecodes
0,12816,12269,1568


## 5. Load reviewed label exclusions

This loads the curated PheCode exclusion list from `final_raw_labels_to_exclude_union_phecode_name_review.txt`
(Naomi's manual review removing labels that should not be used as outcomes,
e.g. screening/symptom codes). Psychosis-related PheCodes are not explicitly removed here unless they appear in that review list.


In [5]:
exclude_rows = []
exclude_path = Path(EXCLUDED_LABELS_PATH)
if exclude_path.exists():
    for line in exclude_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        phecode = line.split('__', 1)[0].strip()
        exclude_rows.append({'excluded_phecode': phecode, 'excluded_disease_label': line})
else:
    print(f'Exclusion file not found: {exclude_path}. No labels excluded by review list.')

excluded_labels_df = pd.DataFrame(exclude_rows).drop_duplicates()
if excluded_labels_df.empty:
    excluded_labels_df = pd.DataFrame(columns=['excluded_phecode', 'excluded_disease_label'])

con.register('excluded_labels_df', excluded_labels_df)
con.execute('CREATE OR REPLACE TEMP TABLE excluded_labels AS SELECT * FROM excluded_labels_df')
print('Review exclusion labels loaded:', len(excluded_labels_df))
display(excluded_labels_df.head())


Review exclusion labels loaded: 250


,excluded_phecode,excluded_disease_label
0,1000.0,1000.0__Burns
1,1001.0,1001.0__Foreign_body_injury
2,1002.0,1002.0__Symptoms_concerning_nutrition__metabol...
3,1004.0,1004.0__Other_signs_and_symptoms_involving_emo...
4,1005.0,1005.0__Other_symptoms


## 6. Map condition events to PheCodes (per row, not yet aggregated)

In [6]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE condition_phecode AS
SELECT DISTINCT
  c.person_id,
  c.visit_occurrence_id,
  CAST(c.condition_start_datetime AS TIMESTAMP) AS condition_start_datetime,
  CAST(c.condition_start_datetime AS DATE) AS condition_date,
  upper(regexp_replace(trim(c.condition_source_value), '[^A-Za-z0-9]', '', 'g')) AS icd10_norm,
  m.phecode,
  m.phenotype,
  m.disease_label
FROM read_parquet({sql_literal(condition_path)}) c
JOIN phecode_map m
  ON upper(regexp_replace(trim(c.condition_source_value), '[^A-Za-z0-9]', '', 'g')) = m.icd10_norm
WHERE c.condition_start_datetime IS NOT NULL
  AND c.condition_source_value IS NOT NULL
  AND regexp_matches(trim(c.condition_source_value), '^[A-Za-z][0-9]')
''')

con.execute('''
CREATE OR REPLACE TEMP TABLE candidate_condition_labels AS
SELECT cp.*
FROM condition_phecode cp
LEFT JOIN excluded_labels e
  ON cp.phecode = e.excluded_phecode
WHERE e.excluded_phecode IS NULL
  AND cp.phecode IS NOT NULL
  AND trim(cp.phecode) <> ''
  AND lower(trim(cp.phecode)) <> 'nan'
''')

print('All mapped condition events:')
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT person_id) AS n_persons,
       COUNT(DISTINCT phecode) AS n_phecodes
FROM condition_phecode
''').to_df())

print('Candidate condition labels after exclusion:')
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT person_id) AS n_persons,
       COUNT(DISTINCT phecode) AS n_phecodes
FROM candidate_condition_labels
''').to_df())


All mapped condition events:


,n_rows,n_persons,n_phecodes
0,218202401,1905218,1167


Candidate condition labels after exclusion:


,n_rows,n_persons,n_phecodes
0,194189703,1851929,1038


## 7. Load precomputed `patient_index` (from notebook 01)

This replaces the original notebook's "select best visit per patient" logic.
The `patient_index` here is fixed by notebook 01 and contains psychosis-risk cases plus sampled non-psychosis controls.


In [7]:
# Load patient_index from notebook 01
con.execute(f'''
CREATE OR REPLACE TEMP TABLE patient_index_raw AS
SELECT
  row_id,
  person_id,
  CAST(index_datetime AS TIMESTAMP) AS index_datetime,
  CAST(index_datetime AS DATE) AS index_date,
  CAST(AGE_AT_INDEX AS INTEGER) AS age_at_index,
  "group" AS cohort_group
FROM read_parquet({sql_literal(PATIENT_INDEX_PATH)})
''')

# Join demographics from person.parquet
con.execute(f'''
CREATE OR REPLACE TEMP TABLE patient_index_with_demo AS
SELECT
  pi.*,
  p.gender_concept_id,
  p.race_concept_id,
  p.ethnicity_concept_id,
  p.gender_source_value,
  p.race_source_value,
  p.ethnicity_source_value,
  p.death_datetime
FROM patient_index_raw pi
LEFT JOIN read_parquet({sql_literal(person_path)}) p USING (person_id)
''')

# Optional filter: require at least one measurement in the lookback window
# if REQUIRE_MEASUREMENT_IN_LOOKBACK:
#     con.execute(f'''
#     CREATE OR REPLACE TEMP TABLE patient_index AS
#     SELECT pi.*
#     FROM patient_index_with_demo pi
#     WHERE EXISTS (
#         SELECT 1
#         FROM read_parquet({sql_literal(measurement_path)}) m
#         WHERE m.person_id = pi.person_id
#           AND m.measurement_datetime >= pi.index_datetime - INTERVAL {int(LOOKBACK_DAYS)} DAY
#           AND m.measurement_datetime <= pi.index_datetime
#           AND m.value_as_number IS NOT NULL
#     )
#     ''')
# else:
#     con.execute('CREATE OR REPLACE TEMP TABLE patient_index AS SELECT * FROM patient_index_with_demo')

print('patient_index after filters:')
display(con.sql('''
SELECT cohort_group, COUNT(*) AS n
FROM patient_index_with_demo
GROUP BY cohort_group
ORDER BY cohort_group
''').to_df())


patient_index after filters:


,cohort_group,n
0,non_psychosis,56778
1,psychosis_risk,56778


## 8. Build labels for each patient (within lookback window before/on index)

In [8]:
# For each patient, labels are PheCodes from condition events with
# condition_date BETWEEN index_date - LOOKBACK_DAYS AND index_date.
# We use the same lookback window as for measurements, for symmetry.

con.execute(f'''
CREATE OR REPLACE TEMP TABLE labels_long_all AS
SELECT DISTINCT
  pi.row_id,
  pi.person_id,
  c.phecode,
  c.phenotype,
  c.disease_label,
  1 AS label
FROM patient_index_with_demo pi
JOIN candidate_condition_labels c
  ON pi.person_id = c.person_id
WHERE  c.condition_start_datetime <= pi.index_datetime
''')

con.execute(f'''
CREATE OR REPLACE TEMP TABLE selected_labels AS
WITH label_counts AS (
    SELECT
      phecode,
      any_value(phenotype) AS phenotype,
      any_value(disease_label) AS disease_label,
      COUNT(DISTINCT person_id) AS n_positive_persons
    FROM labels_long_all
    WHERE phecode IS NOT NULL
      AND trim(phecode) <> ''
      AND lower(trim(phecode)) <> 'nan'
    GROUP BY phecode
    HAVING COUNT(DISTINCT person_id) >= {int(MIN_LABEL_PATIENTS)}
)
SELECT *
FROM label_counts
ORDER BY n_positive_persons DESC
LIMIT {int(MAX_LABELS)}
''')

con.execute('''
CREATE OR REPLACE TEMP TABLE labels_long AS
SELECT l.*
FROM labels_long_all l
JOIN selected_labels s USING (phecode)
''')

print('Selected labels (top 30 by prevalence):')
display(con.sql('SELECT * FROM selected_labels ORDER BY n_positive_persons DESC LIMIT 30').to_df())
display(con.sql('''
SELECT COUNT(*) AS n_label_rows,
       COUNT(DISTINCT person_id) AS n_positive_persons,
       COUNT(DISTINCT phecode) AS n_labels
FROM labels_long
''').to_df())

save_query_csv('SELECT * FROM selected_labels ORDER BY n_positive_persons DESC',
               '02_selected_labels.csv')


Selected labels (top 30 by prevalence):


,phecode,phenotype,disease_label,n_positive_persons
0,300.1,Anxiety disorder,300.1__Anxiety_disorder,18182
1,278.1,Obesity,278.1__Obesity,10324
2,339.0,Other headache syndromes,339.0__Other_headache_syndromes,9733
3,285.0,Other anemias,285.0__Other_anemias,8629
4,465.2,Acute pharyngitis,465.2__Acute_pharyngitis,8398
5,465.0,Acute upper respiratory infections of multiple...,465.0__Acute_upper_respiratory_infections_of_m...,7775
6,530.11,GERD,530.11__GERD,7260
7,591.0,Urinary tract infection,591.0__Urinary_tract_infection,6743
8,614.52,Vaginitis and vulvovaginitis,614.52__Vaginitis_and_vulvovaginitis,6337
9,386.9,Dizziness and giddiness (Light-headedness and ...,386.9__Dizziness_and_giddiness__Light_headedne...,6290


,n_label_rows,n_positive_persons,n_labels
0,430352,67525,994


wrote output/02_selected_labels.csv


PosixPath('output/02_selected_labels.csv')

## 9. Demographic features

In [9]:
con.execute('''
CREATE OR REPLACE TEMP TABLE demographics_features AS
SELECT
  person_id,
  index_datetime,
  cohort_group,
  age_at_index,
  gender_concept_id,
  race_concept_id,
  ethnicity_concept_id,
  gender_source_value,
  race_source_value,
  ethnicity_source_value
FROM patient_index_with_demo
''')

display(con.sql('SELECT * FROM demographics_features LIMIT 10').to_df())
save_table('demographics_features', '04_demographics_features.parquet')


,person_id,index_datetime,cohort_group,age_at_index,gender_concept_id,race_concept_id,ethnicity_concept_id,gender_source_value,race_source_value,ethnicity_source_value
0,10600983,2013-11-25,non_psychosis,34,8532,8527,38003564,Female,White,Not Hispanic or Latino/a/e
1,10608218,2018-06-01,non_psychosis,20,8507,8527,38003563,Male,White,Hispanic or Latino/a/e
2,10621583,2013-02-20,psychosis_risk,22,8507,0,38003564,Male,,Not Hispanic or Latino/a/e
3,10621671,2013-09-27,psychosis_risk,26,8507,8516,38003564,Male,Black or African American,Not Hispanic or Latino/a/e
4,10624731,2013-09-28,non_psychosis,30,8507,8527,38003564,Male,White,Not Hispanic or Latino/a/e
5,10625376,2016-03-07,non_psychosis,20,8507,0,38003563,Male,,Hispanic or Latino/a/e
6,10628821,2014-10-03,non_psychosis,29,8532,8527,38003564,Female,White,Not Hispanic or Latino/a/e
7,10630315,2017-03-06,non_psychosis,29,8532,0,38003564,Female,I Prefer Not To Share,Not Hispanic or Latino/a/e
8,10633182,2020-02-06,psychosis_risk,26,8507,8516,38003564,Male,Black or African American,Not Hispanic or Latino/a/e
9,10634753,2023-01-17,psychosis_risk,30,8532,8527,38003564,Female,White,Not Hispanic or Latino/a/e


wrote output/04_demographics_features.parquet


PosixPath('output/04_demographics_features.parquet')

## 10. Select numeric measurement concepts (prevalence-filtered)

In [10]:
eligible_person_count = con.sql('SELECT COUNT(DISTINCT person_id) FROM patient_index_with_demo').fetchone()[0]
if MIN_MEASUREMENT_PATIENTS is None:
    measurement_patient_threshold = max(1, int(np.ceil(eligible_person_count * MIN_MEASUREMENT_PERSON_PERCENT)))
else:
    measurement_patient_threshold = int(MIN_MEASUREMENT_PATIENTS)

print('Indexed persons:', eligible_person_count)
print('Minimum measurement patient percent:', MIN_MEASUREMENT_PERSON_PERCENT)
print('Minimum measurement patient count:', measurement_patient_threshold)

con.execute(f'''
CREATE OR REPLACE TEMP TABLE all_measurement_names AS
SELECT
  m.measurement_concept_id,
  COALESCE(c.concept_name, 'measurement_' || CAST(m.measurement_concept_id AS VARCHAR)) AS measurement_name,
  any_value(m.measurement_source_value) AS measurement_source_value,
  any_value(m.unit_source_value) AS unit_source_value,
  any_value(c.vocabulary_id) AS vocabulary_id,
  any_value(c.concept_code) AS concept_code,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT m.person_id) AS n_persons
FROM read_parquet({sql_literal(measurement_path)}) m
JOIN patient_index_with_demo pi
  ON m.person_id = pi.person_id
LEFT JOIN read_parquet({sql_literal(concept_path)}) c
  ON m.measurement_concept_id = c.concept_id
WHERE m.measurement_datetime <= pi.index_datetime
  AND m.value_as_number IS NOT NULL
  AND m.measurement_concept_id IS NOT NULL
  AND m.measurement_concept_id <> 0{EXCLUDED_CONCEPT_SQL}
GROUP BY m.measurement_concept_id, c.concept_name
''')

display(con.sql('''
SELECT COUNT(*) AS n_all_measurements_before_filter,
       MIN(n_persons) AS min_persons,
       MAX(n_persons) AS max_persons
FROM all_measurement_names
''').to_df())

# Sanitize concept names into safe column names
def sanitize_feature_name(name):
    s = re.sub(r'[^A-Za-z0-9_]', '_', str(name)).strip('_')
    return s.lower() if s else 'unnamed_measurement'

all_names_df = con.sql('SELECT * FROM all_measurement_names').to_df()
all_names_df['feature_name'] = all_names_df['measurement_name'].apply(sanitize_feature_name)

# Disambiguate any name collisions
seen = {}
final_names = []
for name in all_names_df['feature_name']:
    if name in seen:
        seen[name] += 1
        final_names.append(f'{name}_{seen[name]}')
    else:
        seen[name] = 0
        final_names.append(name)
all_names_df['feature_name'] = final_names

selected_measurement_features = (
    all_names_df[all_names_df['n_persons'] >= measurement_patient_threshold]
    .sort_values(['n_persons', 'n_rows'], ascending=[False, False])
    .head(MAX_MEASUREMENTS)
    .reset_index(drop=True)
)

con.register('selected_measurement_features_df', selected_measurement_features)
con.execute('''
CREATE OR REPLACE TEMP TABLE selected_measurement_features AS
SELECT * FROM selected_measurement_features_df
''')

print('Selected measurement features:', len(selected_measurement_features))
display(selected_measurement_features.head(20))
save_table('selected_measurement_features', '05_selected_measurements.parquet')


Indexed persons: 113556
Minimum measurement patient percent: 0.005
Minimum measurement patient count: 568


,n_all_measurements_before_filter,min_persons,max_persons
0,2228,1,69768


Selected measurement features: 458


,measurement_concept_id,measurement_name,measurement_source_value,unit_source_value,vocabulary_id,concept_code,n_rows,n_persons,feature_name
0,4154790,Diastolic blood pressure,BP Diastolic,,SNOMED,271650006,1957631,69768,diastolic_blood_pressure
1,4152194,Systolic blood pressure,BP Systolic,,SNOMED,271649006,1957608,69768,systolic_blood_pressure_1
2,4108289,Non-invasive mean arterial pressure,NIBP MAP (mmHg) (calculated/READ ONLY),,SNOMED,251074006,1956222,69751,non_invasive_mean_arterial_pressure
3,4093980,Percentage change in weight,Percentage Daily Weight Change,%,SNOMED,248347000,609900,65701,percentage_change_in_weight
4,4099154,Body weight,Weight (lbs),,SNOMED,27113001,1026092,65620,body_weight
5,4086522,Weight change,Daily weight change (gms),,SNOMED,248346009,751464,65445,weight_change
6,4302666,Body temperature,Temp,,SNOMED,386725007,1246848,64297,body_temperature
7,4196147,Peripheral oxygen saturation,SpO2,%,SNOMED,431314004,1524013,62885,peripheral_oxygen_saturation
8,4313591,Respiratory rate,Resp,,SNOMED,86290005,3364442,61055,respiratory_rate
9,3032445,Ideal body weight,IBW/kg (Calculated) Male,kg,LOINC,50064-5,796364,57869,ideal_body_weight


wrote output/05_selected_measurements.parquet


PosixPath('output/05_selected_measurements.parquet')

## 11. Extract latest measurement values per (person, measurement) within lookback

In [11]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE measurement_features_long AS
WITH ranked_measurements AS (
    SELECT
      pi.row_id,
      pi.person_id,
      pi.index_datetime,
      m.measurement_concept_id,
      s.feature_name,
      m.value_as_number,
      m.unit_source_value,
      m.measurement_datetime,
      ROW_NUMBER() OVER (
        PARTITION BY pi.person_id, m.measurement_concept_id
        ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
      ) AS rn
    FROM patient_index_with_demo pi
    JOIN read_parquet({sql_literal(measurement_path)}) m
      ON pi.person_id = m.person_id
    JOIN selected_measurement_features s
      ON m.measurement_concept_id = s.measurement_concept_id
    WHERE m.measurement_datetime <= pi.index_datetime
      AND m.value_as_number IS NOT NULL
      AND m.measurement_concept_id <> 0{EXCLUDED_CONCEPT_SQL}
)
SELECT
  row_id,
  person_id,
  index_datetime,
  measurement_concept_id,
  feature_name,
  value_as_number,
  unit_source_value,
  measurement_datetime
FROM ranked_measurements
WHERE rn = 1
''')

print('Each row is the latest value per (patient, measurement) within lookback.')
display(con.sql('SELECT * FROM measurement_features_long LIMIT 10').to_df())
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT person_id) AS n_persons,
       COUNT(DISTINCT measurement_concept_id) AS n_measurements
FROM measurement_features_long
''').to_df())
save_table('measurement_features_long', '06_measurement_features_long.parquet')


Each row is the latest value per (patient, measurement) within lookback.


,row_id,person_id,index_datetime,measurement_concept_id,feature_name,value_as_number,unit_source_value,measurement_datetime
0,13129,13460402,2022-01-30,4201235,body_surface_area_1,1.92,,2019-11-29 05:00:00
1,13135,13460980,2023-01-31,3006698,p_wave_axis_1,78.00,deg,2022-09-25 11:40:00
2,13141,13462063,2017-12-02,3038058,lymphocytes_100_leukocytes_in_blood_by_manual_...,27.00,%,2017-11-08 19:11:00
3,113184,13462495,2021-08-14,3018171,choriogonadotropin__units_volume__in_serum_or_...,150.00,[iU]/mL,2014-12-10 20:01:00
4,113184,13462495,2021-08-14,3030354,glomerular_filtration_rate_1_73_sq_m_predicted...,60.00,mL/min/1.73.m2,2014-07-25 21:44:00
5,103435,13462589,2024-06-07,3049187,glomerular_filtration_rate_1_73_sq_m_predicted...,60.00,mL/min/1.73.m2,2013-01-18 01:33:00
6,13146,13463915,2016-02-13,4245997,body_mass_index,55.00,,2015-09-24 15:40:00
7,83591,13464443,2021-09-10,3030354,glomerular_filtration_rate_1_73_sq_m_predicted...,60.00,mL/min/1.73.m2,2015-09-23 21:09:00
8,92906,13465282,2021-09-05,4264406,frequency_of_uterine_contraction,2.00,,2018-03-25 05:00:00
9,13157,13466369,2020-04-22,3019170,thyrotropin__units_volume__in_serum_or_plasma_...,0.92,10*-3.[IU]/L,2020-01-09 10:44:00


,n_rows,n_persons,n_measurements
0,5161164,75495,458


wrote output/06_measurement_features_long.parquet


PosixPath('output/06_measurement_features_long.parquet')

## 12. Pivot to wide feature matrix and join demographics

In [12]:
feature_metadata = con.sql('''
SELECT measurement_concept_id, feature_name
FROM selected_measurement_features
ORDER BY n_persons DESC, n_rows DESC
''').fetchall()

feature_exprs = []
for concept_id, feature_name in feature_metadata:
    feature_exprs.append(
        f'MAX(CASE WHEN f.measurement_concept_id = {int(concept_id)} '
        f'THEN f.value_as_number END) AS {qident(feature_name)}'
    )

feature_pivot_sql = ',\n  '.join(feature_exprs) if feature_exprs else 'NULL AS no_measurement_features'

con.execute(f'''
CREATE OR REPLACE TEMP TABLE measurement_features_wide AS
SELECT
  person_id,
  {feature_pivot_sql}
FROM measurement_features_long f
GROUP BY person_id
''')

con.execute('''
CREATE OR REPLACE TEMP TABLE feature_matrix AS
SELECT
  pi.row_id,
  d.person_id,
  d.index_datetime,
  d.cohort_group,
  d.age_at_index,
  d.gender_source_value,
  d.race_source_value,
  d.ethnicity_source_value,
  mw.* EXCLUDE (person_id)
FROM patient_index_with_demo pi
JOIN demographics_features d USING (person_id)
LEFT JOIN measurement_features_wide mw USING (person_id)
ORDER BY pi.row_id
''')

print('feature_matrix: one row per patient, demographics + one column per measurement')
display(con.sql('SELECT COUNT(*) AS n_rows FROM feature_matrix').to_df())
display(con.sql('SELECT * FROM feature_matrix LIMIT 5').to_df())
save_table('feature_matrix', '07_feature_matrix.parquet')


feature_matrix: one row per patient, demographics + one column per measurement


,n_rows
0,113556


,row_id,person_id,index_datetime,cohort_group,age_at_index,gender_source_value,race_source_value,ethnicity_source_value,diastolic_blood_pressure,systolic_blood_pressure_1,non_invasive_mean_arterial_pressure,percentage_change_in_weight,body_weight,weight_change,body_temperature,peripheral_oxygen_saturation,respiratory_rate,ideal_body_weight,body_height_measure,weight_gain,body_mass_index,hemoglobin__mass_volume__in_blood,platelets____volume__in_blood_by_automated_count,mcv__entitic_volume__by_automated_count,erythrocytes____volume__in_blood_by_automated_count,mch__entitic_mass__by_automated_count,body_surface_area_1,erythrocyte_distribution_width__ratio__by_automated_count,leukocytes____volume__in_blood_by_automated_count,mchc__mass_volume__by_automated_count,pain_severity__score___phenx,creatinine__mass_volume__in_serum_or_plasma,glucose__mass_volume__in_serum_or_plasma,urea_nitrogen__mass_volume__in_serum_or_plasma,potassium__moles_volume__in_serum_or_plasma,body_weight_1,sodium__moles_volume__in_serum_or_plasma,chloride__moles_volume__in_serum_or_plasma,emergency_severity_index,heart_rate,diastolic_blood_pressure_1,systolic_blood_pressure,body_mass_index__bmi___ratio,body_height,alanine_aminotransferase__enzymatic_activity_volume__in_serum_or_plasma_by_no_addition_of_p_5__p,bilirubin_total__mass_volume__in_serum_or_plasma,protein__mass_volume__in_serum_or_plasma,globulin__mass_volume__in_plasma,hematocrit__volume_fraction__of_blood_by_automated_count,umbilical_artery_mean_blood_pressure,body_temperature_1,lymphocytes____volume__in_blood_by_automated_count,albumin_globulin__mass_ratio__in_serum_or_plasma,monocytes____volume__in_blood_by_automated_count,basophils____volume__in_blood_by_automated_count,nucleated_erythrocytes_100_leukocytes__ratio__in_blood_by_automated_count,basophils_100_leukocytes_in_blood,glomerular_filtration_rate_1_73_sq_m_predicted__volume_rate_area__in_serum_or_plasma_by_creatinine_based_formula__mdrd,platelet_mean_volume__entitic_volume__in_blood,glomerular_filtration_rate_1_73_sq_m_predicted_among_blacks__volume_rate_area__in_serum__plasma_or_blood_by_creatinine_based_formula__mdrd,...,cardiolipin_igg_ab__units_volume__in_serum_by_immunoassay,atopobium_vaginae_dna__log___volume__in_vaginal_fluid_by_naa_with_probe_detection,epstein_barr_virus_nuclear_ab__units_volume__in_serum,mullerian_inhibiting_substance__mass_volume__in_serum_or_plasma,dna_double_strand_ab__units_volume__in_serum,protein_creatinine__mass_ratio__in_24_hour_urine,megasphaera_sp_dna__log___volume__in_vaginal_fluid_by_naa_with_probe_detection,measles_virus_igg_ab__presence__in_serum,nicotine__mass_volume__in_urine,cotinine__mass_volume__in_urine,tissue_transglutaminase_iga_ab__units_volume__in_serum_by_immunoassay,thiamine__moles_volume__in_serum_or_plasma,fasting_glucose__mass_volume__in_serum_or_plasma,osmolality_of_serum_or_plasma,bias_flow_ventilator,cage_questionnaire,ammonia__moles_volume__in_plasma,glucose__mass_volume__in_serum_or_plasma___1_hour_post_50_g_glucose_po,mumps_virus_igg_ab__presence__in_serum,protein__mass_volume__in_cerebral_spinal_fluid,glucose__mass_volume__in_cerebral_spinal_fluid,cd3_cells_100_cells_in_blood,potassium__moles_volume__in_urine,uterine_contraction_intensity,intrinsic_peep_respiratory_system,cat_dander_ige_ab__units_volume__in_serum,r_r_interval_by_ekg,albumin_creatinine__mass_ratio__in_urine,aspergillus_fumigatus_ige_ab__units_volume__in_serum,alpha_2_globulin__mass_volume__in_serum_or_plasma_by_electrophoresis,alpha_1_globulin__mass_volume__in_serum_or_plasma_by_electrophoresis,gamma_globulin__mass_volume__in_serum_or_plasma_by_electrophoresis,alternaria_alternata_ige_ab__units_volume__in_serum,cladosporium_herbarum_ige_ab__units_volume__in_serum,protein__units_volume__in_urine,heart_rate_intra_arterial_line_by_invasive,vancomycin__mass_volume__in_serum_or_plasma___trough,methylmalonate__moles_volume__in_serum_or_plasma,american_house_dust_mite_ige_ab__units_volume__in_serum,european_house_dust_m

wrote output/07_feature_matrix.parquet


PosixPath('output/07_feature_matrix.parquet')

## 13. Build disease matrix (Y)

In [13]:
selected_label_rows = con.sql('''
SELECT phecode, disease_label
FROM selected_labels
ORDER BY n_positive_persons DESC
''').fetchall()
selected_label_names = [str(x[1]) for x in selected_label_rows]

label_exprs = []
for phecode, disease_label in selected_label_rows:
    label_exprs.append(
        f'MAX(CASE WHEN l.phecode = {sql_literal(phecode)} THEN 1 ELSE 0 END) '
        f'AS {qident(disease_label)}'
    )

label_pivot_sql = ',\n  '.join(label_exprs) if label_exprs else '0 AS no_selected_labels'

con.execute(f'''
CREATE OR REPLACE TEMP TABLE disease_matrix AS
SELECT
  pi.row_id,
  pi.person_id,
  {label_pivot_sql}
FROM patient_index_with_demo pi
LEFT JOIN labels_long l
  ON pi.person_id = l.person_id
 AND pi.row_id = l.row_id
GROUP BY pi.row_id, pi.person_id
ORDER BY pi.row_id
''')

disease_cols = con.sql('DESCRIBE SELECT * FROM disease_matrix').to_df()['column_name'].tolist()
if 'nan' in disease_cols:
    con.execute(f'ALTER TABLE disease_matrix DROP COLUMN {qident("nan")}')
    print('Dropped invalid label column: nan')

print('disease_matrix: one row per patient, one column per PheCode (named phecode__disease_name)')
display(con.sql('SELECT COUNT(*) AS n_rows FROM disease_matrix').to_df())
display(con.sql('SELECT * FROM disease_matrix LIMIT 5').to_df())
save_table('disease_matrix', '08_disease_matrix.parquet')
save_query_csv('''
SELECT phecode, phenotype AS disease_name, disease_label, n_positive_persons
FROM selected_labels
WHERE lower(trim(phecode)) <> 'nan'
ORDER BY n_positive_persons DESC
''', 'disease_names.csv')


disease_matrix: one row per patient, one column per PheCode (named phecode__disease_name)


,n_rows
0,113556


,row_id,person_id,300.1__Anxiety_disorder,278.1__Obesity,339.0__Other_headache_syndromes,285.0__Other_anemias,465.2__Acute_pharyngitis,465.0__Acute_upper_respiratory_infections_of_multiple_or_unspecified_sites,530.11__GERD,591.0__Urinary_tract_infection,614.52__Vaginitis_and_vulvovaginitis,386.9__Dizziness_and_giddiness__Light_headedness_and_vertigo_,300.11__Generalized_anxiety_disorder,619.4__Noninflammatory_disorders_of_vagina,79.0__Viral_infection,401.1__Essential_hypertension,599.3__Dysuria,626.13__Irregular_menstrual_cycle,512.8__Cough,261.4__Vitamin_D_deficiency,427.7__Tachycardia_NOS,626.11__Absent_or_infrequent_menstruation,296.22__Major_depressive_disorder,729.0__Other_disorders_of_soft_tissues,626.0__Disorders_of_menstruation_and_other_abnormal_bleeding_from_female_genital_tract,939.0__Atopic_contact_dermatitis_due_to_other_or_unspecified,313.1__Attention_deficit_hyperactivity_disorder,427.9__Palpitations,296.0__Mood_disorders,706.1__Acne,296.1__Bipolar,280.1__Iron_deficiency_anemias__unspecified_or_not_due_to_blood_loss,687.4__Disturbance_of_skin_sensation,285.1__Acute_posthemorrhagic_anemia,250.2__Type_2_diabetes,476.0__Allergic_rhinitis,619.0__Noninflammatory_female_genital_disorders,112.0__Candidiasis,244.4__Hypothyroidism_NOS,619.3__Noninflammatory_disorders_of_cervix,558.0__Noninfectious_gastroenteritis,300.12__Agorophobia__social_phobia__and_panic_disorder,54.0__Herpes_simplex,90.0__Sexually_transmitted_infections__not_HIV_or_hepatitis_,698.0__Pruritus_and_related_conditions,475.0__Chronic_sinusitis,272.1__Hyperlipidemia,256.4__Polycystic_ovaries,289.4__Lymphadenitis,626.2__Dysmenorrhea,293.1__Swelling__mass__or_lump_in_head_and_neck__Space_occupying_lesion__intracranial_NOS_,614.51__Cervicitis_and_endocervicitis,276.14__Hypopotassemia,216.0__Benign_neoplasm_of_skin,481.0__Influenza,368.0__Visual_disturbances,261.2__Vitamin_B_complex_deficiencies,626.12__Excessive_or_frequent_menstruation,626.1__Irregular_menstrual_cycle_bleeding,594.1__Calculus_of_kidney,...,145.4__Cancer_of_the_gums,442.11__Abdominal_aortic_aneurysm,145.5__Cancer_of_the_mouth_floor,355.0__Complex_regional_central_pain_syndrome,750.0__Digestive_congenital_anomalies,149.1__Cancer_of_oropharynx,750.1__Upper_gastrointestinal_congenital_anomalies,165.1__Cancer_of_bronchus__lung,528.6__Leukoplakia_of_oral_mucosa,624.2__Atrophy_of_female_genital_tract,149.0__Cancer_of_larynx__pharynx__nasal_cavities,378.5__Paralytic_strabismus,732.0__Osteochondropathies,290.11__Alzheimer_s_disease,153.0__Colorectal_cancer,441.1__Acute_vascular_insufficiency_of_intestine,426.31__Right_bundle_branch_block,269.0__Proteinuria,500.2__Pneumoconiosis,259.2__Carcinoid_syndrome,189.4__Malignant_neoplasm_of_other_urinary_organs,446.1__Thromboangiitis_obliterans,536.0__Disorders_of_function_of_stomach,731.0__Osteitis_deformans_and_osteopathies_associated_with_other_disorders_classified_elsewhere,592.3__Urethral_stricture_due_to_infecton,442.2__Aneurysm_of_iliac_artery,523.3__Periodontitis__acute_or_chronic_,446.4__Wegener_s_granulomatosis,530.6__Diverticulum_of_esophagus__acquired,165.0__Cancer_within_the_respiratory_system,513.31__Apnea,172.3__Carcinoma_in_situ_of_skin,702.4__Degenerative_skin_disorders,255.13__Medulloadrenal_hyperfunction,360.0__Disorders_of_the_globe,366.2__Senile_cataract,274.21__Chondrocalcinosis,535.1__Acute_gastritis,379.5__Disorders_of_iris_and_ciliary_body,613.0__Other_nonmalignant_breast_conditions,117.2__Coccidioidomycosis,251.8__Abnormality_of_secretion_of_glucagon_or_gastrin,521.4__Tooth_complications_likely_association_with_other_diseases,505.0__Other_pulmonary_inflamation_or_edema,716.8__Palindromic_rheumatism,731.1__Osteitis_deformans__Paget_s_disease_of_bone_,750.5__Congenital_hypertrophic_pyloric_stenosis,204.4__Multiple_myeloma,149.3__Cancer_of_hypopharynx,573.4__Acute_and_subacute_necrosis_of_liver,381.1__Otitis_media,260.21__Kwashiorkor,379.2__Disorders_of_vitreous_body,521.0__Diseases_of_hard_tissues_of_teeth,528.4__Cysts_of_oral_soft_ti

wrote output/08_disease_matrix.parquet
wrote output/disease_names.csv


PosixPath('output/disease_names.csv')

## 14. Alignment check

In [14]:
alignment_check = con.sql('''
SELECT
  COUNT(*) AS n_rows_checked,
  SUM(CASE WHEN f.row_id = d.row_id AND f.person_id = d.person_id THEN 1 ELSE 0 END) AS n_aligned,
  SUM(CASE WHEN f.row_id <> d.row_id OR f.person_id <> d.person_id THEN 1 ELSE 0 END) AS n_misaligned
FROM feature_matrix f
JOIN disease_matrix d ON f.row_id = d.row_id
''').to_df()
display(alignment_check)


,n_rows_checked,n_aligned,n_misaligned
0,113556,113556.0,0.0


## 15. Summary

In [15]:
n_rows = con.sql('SELECT COUNT(*) FROM feature_matrix').fetchone()[0]
n_disease_rows = con.sql('SELECT COUNT(*) FROM disease_matrix').fetchone()[0]
n_feature_cols = len(con.sql('DESCRIBE SELECT * FROM feature_matrix').to_df())
n_label_cols = len(selected_label_rows)

per_group = con.sql('''
SELECT cohort_group, COUNT(*) AS n
FROM feature_matrix
GROUP BY cohort_group
ORDER BY cohort_group
''').to_df()

# Check whether psychosis-related PheCodes are present in Y (retained unless excluded by the review list).
psychosis_phecodes_check = con.sql('''
SELECT phecode, disease_label, n_positive_persons
FROM selected_labels
WHERE lower(disease_label) LIKE '%psych%'
   OR lower(disease_label) LIKE '%schizo%'
   OR phecode LIKE '295%'
ORDER BY phecode
''').to_df()

summary = pd.DataFrame([{
    'data_path': DATA_PATH,
    'patient_index_path': PATIENT_INDEX_PATH,
    'n_feature_rows': n_rows,
    'n_disease_rows': n_disease_rows,
    'n_feature_columns_including_ids': n_feature_cols,
    'n_label_columns': n_label_cols,
    'min_measurement_person_percent': MIN_MEASUREMENT_PERSON_PERCENT,
    'min_measurement_patients': measurement_patient_threshold,
    'psychosis_related_phecodes_retained_in_Y': True,
    'n_psychosis_related_phecode_columns_in_Y': len(psychosis_phecodes_check),
}])
summary.to_csv(OUTPUT_DIR / '10_dataset_summary.csv', index=False)

print('Per-group counts in final feature_matrix:')
display(per_group)
print()
print('Psychosis-related PheCodes present in Y (retained unless excluded by review list):')
display(psychosis_phecodes_check)
print()
display(summary)


Per-group counts in final feature_matrix:


,cohort_group,n
0,non_psychosis,56778
1,psychosis_risk,56778



Psychosis-related PheCodes present in Y (retained unless excluded by review list):


,phecode,disease_label,n_positive_persons
0,291.4,291.4__Specific_nonpsychotic_mental_disorders_...,8
1,301.1,301.1__Schizoid_personality_disorder,18
2,303.0,303.0__Psychogenic_and_somatoform_disorders,83
3,303.3,303.3__Psychogenic_disorder,229


,data_path,patient_index_path,n_feature_rows,n_disease_rows,n_feature_columns_including_ids,n_label_columns,min_measurement_person_percent,min_measurement_patients,psychosis_related_phecodes_retained_in_Y,n_psychosis_related_phecode_columns_in_Y
0,/home/jupyter/2836994-data2,output/patient_index.parquet,113556,113556,466,994,0.005,568,True,4


## 16. Output manifest

In [16]:
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)


output/02_selected_labels.csv
output/04_demographics_features.parquet
output/05_selected_measurements.parquet
output/06_measurement_features_long.parquet
output/07_feature_matrix.parquet
output/08_disease_matrix.parquet
output/10_dataset_summary.csv
output/control_excluded_persons.parquet
output/disease_names.csv
output/patient_index.parquet
output/psychosis_risk_cohort.parquet
